<a href="https://colab.research.google.com/github/NicoleHun/voice-journal/blob/main/Voice_Journal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
# Cell 1: Install & imports
!pip install google-generativeai gTTS -q

import google.generativeai as genai
from google.colab import files, userdata
import json, re, time
from IPython.display import Audio, display

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.1/155.1 kB 6.1 MB/s eta 0:00:00


In [19]:
# Cell 2: Configure Gemini API key
# Add your key: left sidebar → Secrets (🔑) → New secret → name: GEMINI_API_KEY

genai.configure(api_key=userdata.get('GEMINI_API_KEY'))

# Quick sanity check
model = genai.GenerativeModel('gemini-2.5-flash')
resp = model.generate_content("Reply with: API connected")
print(resp.text)

API connected


In [33]:
# Cell 3: Record audio directly in Colab (no file needed)

from IPython.display import Javascript, display, Audio
from google.colab import output
import base64, time

RECORD_JS = """
async function recordAudio() {
  const duration = 15; // seconds — change this
  const stream = await navigator.mediaDevices.getUserMedia({ audio: true });
  const recorder = new MediaRecorder(stream);
  const chunks = [];

  recorder.ondataavailable = e => chunks.push(e.data);
  recorder.start();
  console.log("Recording...");

  await new Promise(r => setTimeout(r, duration * 1000));
  recorder.stop();
  stream.getTracks().forEach(t => t.stop());

  await new Promise(r => recorder.onstop = r);
  const blob = new Blob(chunks, { type: "audio/wav" });
  const reader = new FileReader();
  reader.readAsDataURL(blob);
  await new Promise(r => reader.onload = r);

  const base64 = reader.result.split(",")[1];
  google.colab.kernel.invokeFunction("save_audio", [base64], {});
}

recordAudio();
"""

def save_audio(b64_data):
    audio_bytes = base64.b64decode(b64_data)
    with open("recorded.wav", "wb") as f:
        f.write(audio_bytes)
    print("Saved as recorded.wav")
    display(Audio("recorded.wav"))

output.register_callback("save_audio", save_audio)
display(Javascript(RECORD_JS))
audio_filename = "recorded.wav"

<IPython.core.display.Javascript object>

Saved as recorded.wav


In [26]:
# Cell 4: Define the extraction prompt (edit this — it's what you'll change in the regression test)

EXTRACT_PROMPT = """
You are a journaling assistant. Listen to this voice journal entry carefully.

Return ONLY valid JSON with this exact structure:
{
  "transcript": "full verbatim transcript of what was said",
  "summary": "2-4 sentence summary in first person, only what was actually said",
  "action_items": ["concrete commitment 1", "concrete commitment 2"]
}

Rules:
- summary must only contain information from the audio. No inference.
- action_items must be explicit commitments, not vague thoughts.
- If there are no action items, return an empty list [].
- Return JSON only. No explanation, no markdown, no backticks.
"""

In [27]:
# Cell 5: Run the pipeline on the uploaded file

def run_pipeline(audio_path, prompt=EXTRACT_PROMPT):
    # Upload to Gemini Files API
    audio_file = genai.upload_file(audio_path, mime_type="audio/wav")

    # Wait for processing
    while audio_file.state.name == "PROCESSING":
        time.sleep(2)
        audio_file = genai.get_file(audio_file.name)

    # Generate
    response = model.generate_content([audio_file, prompt])
    raw = response.text.strip()

    # Strip markdown fences if model adds them anyway
    raw = re.sub(r'^```json\s*', '', raw)
    raw = re.sub(r'```$', '', raw).strip()

    return json.loads(raw)

# Test on your uploaded file
output = run_pipeline(audio_filename)
print(json.dumps(output, indent=2))

{
  "transcript": "I'm looking to um get some grocery and then um potentially go check out the restaurant in mission and also pick up my wallet and then buy tickets. I don't know, just being like very annoying today. I don't really know what to do. It's quite annoying. There's so much work. Yeah, I really don't know. Looks like you have 30 minutes. 30 seconds. Sorry.",
  "summary": "I'm looking to get some groceries, check out a restaurant in Mission, pick up my wallet, and buy tickets. I don't know what to do today, it's quite annoying, and there's so much work.",
  "action_items": [
    "get some grocery",
    "go check out the restaurant in mission",
    "pick up my wallet",
    "buy tickets"
  ]
}


In [34]:
# Cell 6: Build your test dataset
# Upload 6-8 audio files and define expected outputs manually

# Run Cell 3 multiple times to upload each file, or upload all at once:
print("Upload all your test audio files now...")
all_uploads = files.upload()

# Define your test cases — fill in expected values by hand after listening
TEST_CASES = [
    {
        "id": "tc_01",
        "audio_path": "entry_short_vague.wav",       # one of your uploaded filenames
        "description": "Short, vague — mentions needing to 'do something' about work",
        "expected_has_actions": False,                # do you expect action items?
        "expected_summary_topics": ["work"],          # keywords you expect in summary
    },
    {
        "id": "tc_02",
        "audio_path": "recorded.wav",
        "description": "Clear commitments — calls 3 people, books a dentist, sends report",
        "expected_has_actions": True,
        "expected_summary_topics": ["calls", "dentist", "report"],
    },
    {
        "id": "tc_03",
        "audio_path": "entry_emotional.wav",
        "description": "Emotional reflection, no commitments",
        "expected_has_actions": False,
        "expected_summary_topics": [],
    },
    # Add tc_04 through tc_07 here following same pattern
]

print(f"Test cases defined: {len(TEST_CASES)}")

Upload all your test audio files now...


Saving recorded.wav to recorded (1).wav
Saving entry_short_vague.wav to entry_short_vague (1).wav
Saving entry_emotional.wav to entry_emotional (1).wav
Test cases defined: 3


In [35]:
# Cell 7: Run pipeline on all test cases, collect outputs

outputs = []

for tc in TEST_CASES:
    print(f"Running {tc['id']}...")
    try:
        result = run_pipeline(tc["audio_path"])
        outputs.append({
            "id": tc["id"],
            "description": tc["description"],
            "expected_has_actions": tc["expected_has_actions"],
            "expected_topics": tc["expected_summary_topics"],
            "output": result,
            "error": None
        })
        print(f"  ✓ Got {len(result['action_items'])} action items")
    except Exception as e:
        outputs.append({"id": tc["id"], "output": None, "error": str(e)})
        print(f"  ✗ Error: {e}")

print(f"\nDone. {sum(1 for o in outputs if not o['error'])} / {len(outputs)} succeeded.")

Running tc_01...
  ✓ Got 0 action items
Running tc_02...
  ✓ Got 4 action items
Running tc_03...
  ✓ Got 0 action items

Done. 3 / 3 succeeded.


In [36]:
# Cell 8: Level 1 — Heuristic evals

def heuristic_eval(entry):
    out = entry["output"]
    results = {}

    if out is None:
        return {"error": "pipeline_failed"}

    # 1. Schema check
    results["schema_ok"] = (
        isinstance(out.get("summary"), str) and
        isinstance(out.get("action_items"), list) and
        isinstance(out.get("transcript"), str)
    )

    # 2. Summary length (20–150 words)
    word_count = len(out["summary"].split())
    results["summary_length_ok"] = 20 <= word_count <= 150
    results["summary_word_count"] = word_count

    # 3. Hallucination check — each action item must share words with transcript
    transcript_words = set(out["transcript"].lower().split())
    hallucinated = []
    for item in out["action_items"]:
        item_words = set(item.lower().split())
        if not item_words & transcript_words:
            hallucinated.append(item)
    results["no_hallucination"] = len(hallucinated) == 0
    results["hallucinated_items"] = hallucinated

    # 4. Empty action items flag (not a fail, just visibility)
    results["has_action_items"] = len(out["action_items"]) > 0

    # 5. Expected actions match
    if entry.get("expected_has_actions") is not None:
        results["actions_match_expectation"] = (
            entry["expected_has_actions"] == results["has_action_items"]
        )

    return results

# Run and display
print("=" * 60)
print("LEVEL 1: HEURISTIC EVAL RESULTS")
print("=" * 60)

heuristic_results = []
for entry in outputs:
    r = heuristic_eval(entry)
    heuristic_results.append({"id": entry["id"], "results": r})

    status = "PASS" if all(
        v for k, v in r.items()
        if k not in ("summary_word_count", "hallucinated_items",
                     "has_action_items", "actions_match_expectation")
        and isinstance(v, bool)
    ) else "FAIL"

    print(f"\n[{status}] {entry['id']} — {entry['description']}")
    for k, v in r.items():
        print(f"  {k}: {v}")

LEVEL 1: HEURISTIC EVAL RESULTS

[FAIL] tc_01 — Short, vague — mentions needing to 'do something' about work
  schema_ok: True
  summary_length_ok: False
  summary_word_count: 15
  no_hallucination: True
  hallucinated_items: []
  has_action_items: False
  actions_match_expectation: True

[PASS] tc_02 — Clear commitments — calls 3 people, books a dentist, sends report
  schema_ok: True
  summary_length_ok: True
  summary_word_count: 30
  no_hallucination: True
  hallucinated_items: []
  has_action_items: True
  actions_match_expectation: True

[PASS] tc_03 — Emotional reflection, no commitments
  schema_ok: True
  summary_length_ok: True
  summary_word_count: 30
  no_hallucination: True
  hallucinated_items: []
  has_action_items: False
  actions_match_expectation: True


In [37]:
# Cell 9: Define the judge prompt (separate from extraction prompt)

JUDGE_PROMPT = """
You are a strict quality evaluator for an AI journaling assistant.

Given a voice journal transcript and the AI's output (summary + action items),
score the output on four dimensions. Score each 1–5 where:
  1 = very poor, 2 = poor, 3 = acceptable, 4 = good, 5 = excellent

Dimensions:
- faithfulness: Does the summary contain ONLY information from the transcript?
  (Penalise any invented or inferred content not said by the speaker)
- completeness: Does the summary capture all key points from the transcript?
- conciseness: Is the summary appropriately brief without being too thin?
- action_precision: Do the action items reflect ONLY explicit commitments?
  (Penalise vague thoughts listed as actions, or missed explicit commitments)

IMPORTANT calibration:
- Score 5 only if the output is near-perfect on that dimension
- Score 1-2 if there is a clear, specific failure
- Most outputs should score 2-4
- Be critical. Do not be generous to avoid conflict.

Return ONLY valid JSON:
{
  "scores": {
    "faithfulness": <int>,
    "completeness": <int>,
    "conciseness": <int>,
    "action_precision": <int>
  },
  "reasoning": {
    "faithfulness": "<one sentence>",
    "completeness": "<one sentence>",
    "conciseness": "<one sentence>",
    "action_precision": "<one sentence>"
  },
  "overall": <int>
}
"""

In [39]:
# Cell 10: Run LLM-as-judge on all outputs

def run_judge(entry, judge_prompt=JUDGE_PROMPT):
    out = entry["output"]
    if out is None:
        return None

    judge_input = f"""
TRANSCRIPT:
{out['transcript']}

AI SUMMARY:
{out['summary']}

AI ACTION ITEMS:
{json.dumps(out['action_items'], indent=2)}
"""

    response = model.generate_content([judge_prompt, judge_input])
    raw = response.text.strip()
    raw = re.sub(r'^```json\s*', '', raw)
    raw = re.sub(r'```$', '', raw).strip()
    return json.loads(raw)


print("=" * 60)
print("LEVEL 2: LLM-AS-JUDGE RESULTS")
print("=" * 60)

judge_results = []
for entry in outputs:
    print(f"\nJudging {entry['id']}...")
    try:
        r = run_judge(entry)
        judge_results.append({"id": entry["id"], "judge": r})

        scores = r["scores"]
        avg = sum(scores.values()) / len(scores)
        print(f"  Overall: {r['overall']}/5  |  Avg dimension: {avg:.1f}")
        for dim, score in scores.items():
            print(f"  {dim}: {score}/5 — {r['reasoning'][dim]}")
    except Exception as e:
        judge_results.append({"id": entry["id"], "judge": None, "error": str(e)})
        print(f"  Error: {e}")

LEVEL 2: LLM-AS-JUDGE RESULTS

Judging tc_01...
  Overall: 5/5  |  Avg dimension: 4.8
  faithfulness: 5/5 — The summary is a near-verbatim extraction of the key phrases, containing no invented or inferred content.
  completeness: 5/5 — Given the brevity and lack of detail in the transcript, the summary successfully captures all stated key points.
  conciseness: 4/5 — The summary is appropriately brief for the extremely short input, removing only filler words without losing meaning.
  action_precision: 5/5 — The AI correctly identified that there are no specific, actionable commitments in the transcript, hence returning an empty list of actions.

Judging tc_02...
  Overall: 4/5  |  Avg dimension: 4.2
  faithfulness: 3/5 — The summary introduces first-person framing like 'I need to' and 'I plan to' which were not explicitly stated by the speaker in the transcript's imperative form.
  completeness: 5/5 — All four distinct tasks mentioned in the transcript are fully present in the summary.

In [ ]:
# Cell 11: Print consolidated score table

print("\n" + "=" * 60)
print("SCORE SUMMARY")
print("=" * 60)
print(f"{'ID':<10} {'faithful':>9} {'complete':>9} {'concise':>9} {'action':>9} {'overall':>9}")
print("-" * 60)

for jr in judge_results:
    if jr.get("judge"):
        s = jr["judge"]["scores"]
        print(f"{jr['id']:<10} {s['faithfulness']:>9} {s['completeness']:>9} "
              f"{s['conciseness']:>9} {s['action_precision']:>9} "
              f"{jr['judge']['overall']:>9}")

print("-" * 60)
print("\nHeuristic failures:")
for hr in heuristic_results:
    fails = [k for k, v in hr["results"].items()
             if isinstance(v, bool) and not v
             and k not in ("has_action_items",)]
    if fails:
        print(f"  {hr['id']}: {', '.join(fails)}")